# 01 · Unified data center list
We merge four public sources (Cleanview Virginia sample, Cleanview national table, Brockovich, IM3 Atlas) into one list of US data centers of at least 10 MW, with the same columns for every site and no duplicates.
Output: `data/clean/dc_unified.csv`, read by notebooks 02, 03 and 04 (Section 2.1.1 of the thesis).

Setup. `MIN_MW` is the minimum facility power kept in the list.

In [1]:
import pandas as pd
import numpy as np
from utils import RAW, CLEAN, distance_km

MIN_MW = 10

## 1. Load the four sources
The raw tables stay untouched; we work on copies.

In [2]:
# read the data
va_raw = pd.read_csv(RAW / "dc/cleanview_virginia.csv")
nat_raw = pd.read_csv(RAW / "dc/cleanview_national.csv")
br_raw = pd.read_csv(RAW / "dc/brockovich.csv")
im_raw = pd.read_csv(RAW / "dc/im3_atlas.csv", dtype={"id": str})

# create tables
raw_tables = {"Virginia": va_raw, "National": nat_raw, "Brockovich": br_raw, "IM3": im_raw}

# print info for each source
for name, df in raw_tables.items():
    print(name, "—", len(df), "rows and", len(df.columns), "columns")

Virginia — 748 rows and 18 columns
National — 100 rows and 17 columns
Brockovich — 474 rows and 25 columns
IM3 — 1479 rows and 13 columns


## 2. Look at the data before merging
Missing values and types of every source. A missing MW means "unknown", not "small".

In [3]:
for name, df in raw_tables.items():
    print("\n" + name)
    profile = pd.DataFrame({"type": df.dtypes, "missing": df.isna().sum(),
                            "missing_percent": (100 * df.isna().mean()).round(1)})
    print(profile.to_string())
    profile.to_csv(CLEAN / (name.lower() + "_profile.csv"))


Virginia
                               type  missing  missing_percent
id                            int64        0              0.0
project_name                 object        0              0.0
developer                    object        3              0.4
status                       object        0              0.0
operating_year              float64      156             20.9
total_facility_capacity_mw  float64       67              9.0
it_capacity_mw              float64       89             11.9
estimated_investment_usd    float64       67              9.0
building_square_footage     float64       89             11.9
county                       object        0              0.0
state                        object        0              0.0
latitude                    float64        5              0.7
longitude                   float64        5              0.7
tenant                       object      695             92.9
end_user                    float64      748            100.

Coverage by state and the labels each source uses for status and precision: the sources are not interchangeable (Virginia only, first 100 national rows, projects, buildings).

In [4]:
# each source and each state 
coverage = pd.DataFrame({"Virginia": va_raw.state.value_counts(),
                         "National": nat_raw.State.value_counts(),
                         "Brockovich": br_raw.state.value_counts(),
                         "IM3": im_raw.state_abb.value_counts()}).fillna(0).astype(int)

# display & save as csv
print(coverage.to_string())
coverage.to_csv(CLEAN / "raw_state_coverage.csv")

# show DC status for each source
for label, values in {"Virginia status": va_raw.status, "National status": nat_raw.Status,
                      "Brockovich status": br_raw.status, "Brockovich precision": br_raw.precision,
                      "Brockovich confidence": br_raw.confidence, "IM3 unit": im_raw.type,
                      "National operator": nat_raw.Developer}.items():
    print("\n" + label)
    print(values.value_counts(dropna=False).to_string())

    Virginia  National  Brockovich  IM3
AK         0         0           1    0
AL         0         0           6   11
AR         0         0           4    2
AZ         0        13          21   65
CA         0         1          17  112
CO         0         0           5   20
CT         0         0           1    5
DC         0         0           0    1
DE         0         0           1    0
FL         0         0           5   16
GA         0         2          23   38
HI         0         0           1    0
IA         0         0           8   64
ID         0         0           2    5
IL         0        13          16   50
IN         0         0          14    5
KS         0         0           4    7
KY         0         0           5    5
LA         0         0           7    1
MA         0         0           2    3
MD         0         4           6   13
ME         0         0           3    2
MI         0         0           6   10
MN         0         0           5   11


Power and floor-area distributions. Some IM3 buildings appear twice with different counties: we keep one copy and leave their county blank.

In [5]:
def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(r"[,$]", "", regex=True), errors="coerce")

# info on facility's MW 
for name, values in {"Virginia MW": va_raw.total_facility_capacity_mw,
                     "National MW": nat_raw["Facility Power (MW)"],
                     "Brockovich MW": br_raw.capacity_mw, "IM3 square feet": im_raw.sqft}.items():
    print("\n" + name)
    print(to_num(values).describe().round(1).to_string())

im_county_duplicates = im_raw[im_raw.id.duplicated(keep=False)].copy()
im_county_duplicates.to_csv(CLEAN / "im3_repeated_county_records.csv", index=False)
print("\nRepeated IM3 ID rows beyond the first:", im_raw.id.duplicated().sum())

# Keep all county evidence, but only one copy of each object for matching/estimation.
im = im_raw.drop_duplicates("id").copy()
im["county_ambiguous"] = im.id.isin(im_county_duplicates.id)
im.loc[im.county_ambiguous, "county"] = np.nan


Virginia MW
count     681.0
mean       79.7
std       149.2
min         0.8
25%        44.4
50%        64.4
75%        74.8
max      2400.0

National MW
count     100.0
mean      116.2
std       243.7
min         2.0
25%        70.0
50%        83.0
75%       108.5
max      2457.0

Brockovich MW
count      260.0
mean       828.4
std       1253.9
min          2.0
25%        168.0
50%        400.0
75%       1000.0
max      10000.0

IM3 square feet
count        1374.0
mean       604174.1
std       2452628.0
min           204.0
25%         60050.2
50%        135001.0
75%        242613.2
max      26612163.0

Repeated IM3 ID rows beyond the first: 5


## 3. A common format
Every source gets the same columns. Status labels are harmonized and, when only IT power is given, facility power = IT × 1.3.

In [6]:
# define dict of operating status + column names 
STATUS = {"operating": "operating", "operational": "operating",
          "under construction": "under construction", "under_construction": "under construction",
          "planned": "planned", "proposed": "planned", "permitted": "planned",
          "canceled": "cancelled", "cancelled": "cancelled"}
COLS = ["name", "operator", "status", "lat", "lon", "state", "county", "facility_mw",
        "mw_source", "operating_year", "source", "source_row", "announced",
        "raw_confidence", "precision", "type", "sqft", "county_ambiguous"]

# standardisation function
def standardise(df, facility, it, source):
    df = df.copy()
    fac = to_num(facility)
    it_mw = to_num(it) * 1.3
    df["facility_mw"] = fac.fillna(it_mw)
    df["mw_source"] = np.where(fac.notna(), "reported", np.where(it_mw.notna(), "IT x 1.3", "missing"))
    df["status"] = df.status.str.lower().map(STATUS).fillna("unknown")
    df["source"] = source
    df["source_row"] = df.index
    for col in COLS:
        if col not in df:
            df[col] = np.nan
    return df[COLS]

We apply the format to each source. The 24 national projects already in the Virginia sample are removed (and saved in `national_overlap.csv`). IM3 has no status or MW.

In [7]:
# virginia and national Cleanview data
va = va_raw.rename(columns={"project_name": "name", "developer": "operator", "latitude": "lat", "longitude": "lon"})
nat = nat_raw.rename(columns={"Project Name": "name", "Developer": "operator", "Status": "status",
    "Latitude": "lat", "Longitude": "lon", "State": "state", "County": "county", "Year": "operating_year"})

# In these frozen files, all 24 repeat names also agree on state/operator/coordinates.
# Keep the removed observations for inspection rather than deleting the evidence.
national_overlap = nat[nat.name.isin(va.name)].copy()
national_overlap.to_csv(CLEAN / "national_overlap.csv", index=False)
nat = nat[~nat.name.isin(va.name)].copy()
cv_va = standardise(va, va.total_facility_capacity_mw, va.it_capacity_mw, "cleanview_va")
cv_nat = standardise(nat, nat["Facility Power (MW)"], nat["IT Capacity (MW)"], "cleanview_national")

# merge virginia and national cleanview into one 
cleanview = pd.concat([cv_va, cv_nat], ignore_index=True)

# brockovich 
br = br_raw.rename(columns={"latitude": "lat", "longitude": "lon", "confidence": "raw_confidence"})
brock = standardise(br, br.capacity_mw, pd.Series(np.nan, index=br.index), "brockovich")

# IM3
im = im.drop(columns="state").rename(columns={"state_abb": "state"})
im["status"] = "unknown"
im3 = standardise(im, pd.Series(np.nan, index=im.index), pd.Series(np.nan, index=im.index), "im3")
im3 = im3.reset_index(drop=True)

# info on each source 
print({"Cleanview": len(cleanview), "Brockovich": len(brock), "IM3": len(im3)})
print("National repeat observations:", len(national_overlap))
print("Missing coordinates:", cleanview[["lat", "lon"]].isna().any(axis=1).sum())

{'Cleanview': 824, 'Brockovich': 474, 'IM3': 1474}
National repeat observations: 24
Missing coordinates: 6


## 4. Find the same site in different sources
Two records are the same site if they have the same operator and are less than `km` apart, or if they are less than 300 m apart whatever the operator. Every match is saved for review.

In [8]:
ALIAS = {"aws": "amazon", "amazon web services": "amazon", "facebook": "meta",
         "alphabet": "google", "quality technology services": "qts", "aligned data centers": "aligned"}

def operator_key(s):
    return s.fillna("").str.lower().str.strip().replace(ALIAS)

def find_matches(new, base, km):
    """For each new record: the closest base record that is the same site, or -1."""
    new_key, base_key = operator_key(new.operator).values, operator_key(base.operator).values
    matches = []
    for i, row in new.iterrows():
        d = distance_km(row.lat, row.lon, base.lat.values, base.lon.values)
        same_op = (base_key == new_key[i]) & (new_key[i] != "")
        ok = (same_op & (d < km)) | (d < 0.3)
        matches.append(base.index[ok][np.argmin(d[ok])] if ok.any() else -1)
    return np.array(matches)

match_rows = []

def record_matches(new, base, matches):
    for i, target in enumerate(matches):
        if target < 0:
            continue
        row, kept = new.iloc[i], base.loc[target]
        match_rows.append({"incoming_source": row.source, "incoming_row": row.source_row,
            "incoming_name": row["name"], "kept_source": kept.source, "kept_row": kept.source_row,
            "kept_name": kept["name"], "incoming_mw": row.facility_mw, "kept_mw": kept.facility_mw,
            "incoming_status": row.status, "kept_status": kept.status, "incoming_type": row.type,
            "incoming_announced": row.announced, "incoming_confidence": row.raw_confidence,
            "distance_km": distance_km(row.lat, row.lon, kept.lat, kept.lon)})

Cleanview is the reference. Brockovich is compared with it (1.5 km), then IM3 with the combined table (1 km). A matched record is not added again: it only counts as a confirmation of the site.

In [9]:
# start from cleanview data 
sites = cleanview.copy()
sites["sources"] = "cleanview"

# match brockovich data to cleanview 
brock_match = find_matches(brock, sites, km=1.5)
record_matches(brock, sites, brock_match)
for target in np.unique(brock_match[brock_match >= 0]):
    sites.loc[target, "sources"] += ";brockovich"
sites = pd.concat([sites, brock[brock_match < 0].assign(sources="brockovich")], ignore_index=True)

# match IM3 data to cleanview + brockovich 
im3["match"] = find_matches(im3, sites, km=1.0)
record_matches(im3, sites, im3.match.values)
for target in np.unique(im3.loc[im3.match >= 0, "match"]):
    sites.loc[target, "sources"] += ";im3"
    
match_review = pd.DataFrame(match_rows)
match_review.to_csv(CLEAN / "matches_to_review.csv", index=False)

# print results
print("Brockovich matched / appended:", (brock_match >= 0).sum(), (brock_match < 0).sum())
print("IM3 matched / unmatched:", (im3.match >= 0).sum(), (im3.match < 0).sum())
print(match_review.head(10).to_string(index=False))

Brockovich matched / appended:

 15 459
IM3 matched / unmatched: 451 1023
incoming_source  incoming_row               incoming_name        kept_source  kept_row                    kept_name  incoming_mw  kept_mw    incoming_status        kept_status incoming_type incoming_announced incoming_confidence  distance_km
     brockovich            12        Aligned PHX-01/02/03 cleanview_national        37 Aligned Phoenix - Building B          NaN     16.0          operating          operating           NaN                NaN            reported     0.092774
     brockovich            95              Aligned ORD-01 cleanview_national         9       Aligned Chicago ORD-01          NaN     50.0          operating          operating           NaN                NaN            reported     0.012997
     brockovich           270              Aligned NEO-01 cleanview_national        31               Aligned NEO-01         96.0     83.0 under construction          operating           NaN                NaN            reported    

## 5. MW for IM3 buildings
IM3 buildings that match a site with reported MW give a ratio of MW per million square feet (median about 315). We use it to estimate the power of the IM3 buildings that no other source lists.

In [10]:
pairs = im3.loc[im3.match >= 0, ["sqft", "type", "match"]].join(
    sites[["facility_mw", "mw_source", "state", "status"]], on="match")
pairs = pairs[(pairs.mw_source == "reported") & (pairs.sqft > 0) & (pairs.type == "building")].copy()
pairs["ratio"] = pairs.facility_mw / (pairs.sqft / 1e6)
ratio = pairs.ratio.median()
print("Calibration pairs:", len(pairs), "Unique retained records:", pairs.match.nunique())
print(pairs.groupby(["state", "status"]).size().to_string())
print(pairs.ratio.describe().round(2).to_string())
pairs.to_csv(CLEAN / "calibration_pairs.csv", index=False)

im3_new = im3[(im3.match < 0) & (im3.type == "building") & (im3.sqft > 0)].copy()
im3_new["facility_mw"] = im3_new.sqft / 1e6 * ratio
im3_new["mw_source"] = "estimated from floor area"
im3_new["sources"] = "im3"
im3_new.to_csv(CLEAN / "estimated_candidates.csv", index=False)
for label, table in {"Without estimates": sites,
                     "With estimates": pd.concat([sites, im3_new], ignore_index=True)}.items():
    kept = table[(table.status != "cancelled") & (table.facility_mw >= MIN_MW)]
    print(label, "—", len(kept), "rows;", round(kept.facility_mw.sum(), 1), "MW")

assert pd.notna(ratio), "No valid calibration data"
sites = pd.concat([sites, im3_new[COLS + ["sources"]]], ignore_index=True)

Calibration pairs: 298 Unique retained records: 278
state  status            
AZ     operating               2
       under construction      1
IL     operating               2
NV     operating               1
OH     operating               3
       under construction      3
OR     operating               1
       planned                 2
TX     operating               4
       under construction      1
VA     operating             265
       planned                 2
       under construction     11


count     298.00
mean      499.52
std       704.10
min        18.19
25%       228.91
50%       315.33
75%       494.53
max      4738.75
Without estimates — 964 rows; 271722.2 MW
With estimates — 1630 rows; 310430.1 MW


## 6. Filter
We keep sites of at least 10 MW that are not cancelled, and save every excluded record with its reason.

In [11]:
sites["rumored"] = sites.raw_confidence.eq("rumored")
sites["n_sources"] = sites.sources.str.count(";") + 1
sites["confidence"] = np.where(sites.mw_source.isin(["reported", "IT x 1.3"]) & ~sites.rumored, "medium", "low")
sites["filter_reason"] = np.select(
    [sites.status.eq("cancelled"), sites.facility_mw.isna(), sites.facility_mw.lt(MIN_MW)],
    ["cancelled", "missing MW", "below threshold"], default="included")
ledger = sites.groupby(["source", "filter_reason"]).size().unstack(fill_value=0)
print(ledger.to_string())
ledger.to_csv(CLEAN / "filter_ledger.csv")
sites.to_csv(CLEAN / "all_candidates.csv", index=False)
sites[sites.filter_reason != "included"].to_csv(CLEAN / "excluded_candidates.csv", index=False)
dc = sites[sites.filter_reason == "included"].copy()
print("Candidates / included:", len(sites), len(dc))

filter_reason       below threshold  cancelled  included  missing MW
source                                                              
brockovich                        3         19       243         194
cleanview_national                3          0        73           0
cleanview_va                     31          5       648          64
im3                             169          0       666           0


Candidates / included: 2118 1630


## 7. PJM zone
Each site gets its PJM zone from the county-to-zone table (blank = outside PJM or county not found).

In [12]:
def county_key(s):
    return s.fillna("").str.lower().str.replace(" county", "", regex=False).str.strip()
zones = pd.read_csv(RAW / "manual/county_to_pjm_zone.csv")
zones["key"] = zones.state + "|" + county_key(zones.county)
assert zones.key.is_unique, "Repeated county lookup keys"
dc["zone_lookup"] = (dc.state + "|" + county_key(dc.county)).map(zones.set_index("key").pjm_zone)
dc["pjm_zone"] = dc.zone_lookup.replace("NON-PJM", np.nan)
dc["zone_result"] = np.where(dc.zone_lookup.eq("NON-PJM"), "known non-PJM",
                            np.where(dc.zone_lookup.notna(), "county match", "no lookup match"))
print(dc.zone_result.value_counts().to_string())

zone_result
county match       850
no lookup match    726
known non-PJM       54


## 8. Provisional list
Sites sorted by size, with a sequential id.

In [13]:
dc["has_operating_year"] = dc.status.eq("operating") & pd.to_numeric(dc.operating_year, errors="coerce").between(1900, 2026)
print("Operating with a reported year:", dc.has_operating_year.sum(), "of", len(dc))
print(dc.groupby(["state", "source"]).size().unstack(fill_value=0).to_string())
dc = dc.sort_values("facility_mw", ascending=False).reset_index(drop=True)
dc.insert(0, "dc_id", range(1, len(dc) + 1))
dc.to_csv(CLEAN / "dc_unified_provisional.csv", index=False)
print("Saved", len(dc), "provisional rows and", round(dc.facility_mw.sum(), 1), "MW")

Operating with a reported year: 334 of 1630
source  brockovich  cleanview_national  cleanview_va  im3
state                                                    
AK               1                   0             0    0
AL               3                   0             0    5
AR               1                   0             0    1
AZ              10                  13             0   29
CA              11                   1             0   76
CO               5                   0             0   14
CT               0                   0             0    3
DE               1                   0             0    0
FL               3                   0             0    6
GA              12                   2             0   22
IA               1                   0             0   48
ID               0                   0             0    2
IL               9                  13             0   26
IN               9                   0             0    2
KS               1          

## 9. Official list
The list used in the thesis: mapped IM3 buildings count as operating sites, and confidence is high when two or more sources report the site, low when MW is estimated or the project is only rumored, medium otherwise. Result: 1,630 sites (1,028 operating, 241 under construction, 361 planned).

In [14]:
# Official list used by notebooks 02-04: the original rules, applied on top of the cleaned pipeline above.
official = dc.copy()
official["source"] = official.source.replace({"cleanview_va": "cleanview", "cleanview_national": "cleanview"})
official.loc[official.source.eq("im3") & official.status.eq("unknown"), "status"] = "operating"   # mapped IM3 buildings already exist

# An IM3 object listed in two counties keeps its PJM zone when both counties are in the same zone
dup = im_county_duplicates.assign(key=im_county_duplicates.state_abb + "|" + county_key(im_county_duplicates.county))
dup["zone"] = dup.key.map(zones.set_index("key").pjm_zone)
one_zone = dup.groupby("id").zone.agg(lambda z: z.iloc[0] if z.nunique(dropna=False) == 1 else np.nan)
is_amb = official.source.eq("im3") & official.county_ambiguous.eq(True)
ids = im_raw.loc[official.loc[is_amb, "source_row"].astype(int), "id"].values
official.loc[is_amb, "pjm_zone"] = pd.Series(ids).map(one_zone).replace("NON-PJM", np.nan).values

official["confidence"] = np.select(
    [official.n_sources >= 2, official.mw_source.eq("estimated from floor area") | official.rumored],
    ["high", "low"], default="medium")   # high = found in two or more sources
OFFICIAL_COLS = ["dc_id", "name", "operator", "status", "lat", "lon", "state", "county", "facility_mw", "mw_source",
                 "operating_year", "source", "sources", "n_sources", "confidence", "pjm_zone"]
official = official[OFFICIAL_COLS]
official.to_csv(CLEAN / "dc_unified.csv", index=False)
print("Saved", len(official), "rows to dc_unified.csv")
print(pd.crosstab(official.source, official.status).to_string())
print(official.confidence.value_counts().to_string())

Saved 1630 rows to dc_unified.csv
status      operating  planned  under construction
source                                            
brockovich         23      114                 106
cleanview         339      247                 135
im3               666        0                   0
confidence
medium    689
low       671
high      270
